# Tarea 2 : Creación de Agentes con Hugging Face - Personalizacion agente

## Angie Yesenia Giraldo Galeano

## Cesar Augusto Florez Castaño

## Jhonatan Rojas Diaz

**INTRODUCCIÓN**

Cuando hablamos de agentes inteligentes, la diferencia entre uno útil y uno que simplemente "responde texto" está en las herramientas que tiene a su disposición. En este ejercicio partimos del agente básico que ya teníamos funcionando en Hugging Face Spaces y le dimos verdaderos superpoderes: la capacidad de calcular áreas geométricas y de consultar el clima real de cualquier ciudad del mundo. Para esto usamos la librería smolagents, que nos permite definir herramientas de forma sencilla y conectarlas al agente para que él mismo decida cuándo y cómo usarlas según la pregunta del usuario.


Además de agregar nuevas habilidades, también intervenimos la forma en que el agente entrega sus respuestas. Modificamos la clase FinalAnswerTool para que cada respuesta tenga un formato personalizado: un prefijo que identifica al agente, el conteo de caracteres del mensaje y una firma del equipo al final. Aunque parece un detalle pequeño, este cambio muestra algo importante: en el desarrollo de agentes no solo importa qué hace el sistema, sino también cómo lo comunica.

In [1]:
pip install smolagents gradio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#se importan librerias necesarias para el funcionamiento del agente.

import random
import requests
from smolagents import CodeAgent, InferenceClientModel, tool, FinalAnswerTool
import gradio as gr
import math
import os
from dotenv import load_dotenv
from huggingface_hub import login



In [3]:

# Carga las variables de entorno del archivo .env
load_dotenv()

# Lee el token desde las variables de entorno
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Autenticado con Hugging Face")
else:
    print("HF_TOKEN no encontrado. Verifica tu archivo .env")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Autenticado con Hugging Face


In [4]:
#Herramienta 1: Calculadora de áreas de figuras geométricas
@tool
def calcular_area(figura: str, valor1: float, valor2: float = 0) -> str: #Se crea función para calcular el área de diferentes figuras geométricas con dos valores.
    
    #Instrucciones para el agente sobre la herramienta de calcular áreas.
    
    """
    Calcula el área de una figura geométrica.

    Args:
        figura: Nombre de la figura. Puede ser 'rectangulo', 'triangulo' o 'circulo'.
        valor1: Para rectángulo y triángulo es la base. Para círculo es el radio.
        valor2: Para rectángulo es la altura. Para triángulo es la altura. No aplica al círculo.

    Returns:
        Un string con el resultado del área calculada.
    """
    figura = figura.lower() # Convertimos el nombre de la figura a minúsculas para evitar problemas de mayúsculas y minúsculas
    
    if figura == "rectangulo":
        area = valor1 * valor2 # Para el rectángulo, el área se calcula multiplicando la base (valor1) por la altura (valor2)
        return f"El área del rectángulo con base {valor1} y altura {valor2} es: {area:.2f}"

    elif figura == "triangulo":
        area = (valor1 * valor2) / 2 # Para el triángulo, el área se calcula multiplicando la base (valor1) por la altura (valor2) y dividiendo entre 2
        return f"El área del triángulo con base {valor1} y altura {valor2} es: {area:.2f}"

    elif figura == "circulo":
        area = math.pi * (valor1 ** 2) # Para el círculo, el área se calcula multiplicando pi por el radio (valor1) al cuadrado
        return f"El área del círculo con radio {valor1} es: {area:.2f}"

    else:
        return f"Figura '{figura}' no reconocida. Usa: rectangulo, triangulo o circulo."

In [5]:

#herramienta 2: Obtener el clima de una ciudad usando la API gratuita de wttr.in
@tool
def obtener_clima(ciudad: str) -> str: #Función para obtener el clima actual de una ciudad usando la API gratuita de wttr.in.
    
    #Instrucciones para el agente sobre la herramienta de obtener clima.
    
    """
    Obtiene el clima actual de una ciudad usando la API gratuita de wttr.in.

    Args:
        ciudad: Nombre de la ciudad en español o inglés (ej: 'Medellin', 'Bogota', 'London').

    Returns:
        Un string con la información del clima actual de la ciudad.
    """
    try:
        # wttr.in es una API pública y gratuita, no requiere clave
        url = f"https://wttr.in/{ciudad}?format=3&lang=es"
        response = requests.get(url, timeout=5)

        if response.status_code == 200:
            return f"🌤️ Clima en {ciudad}: {response.text.strip()}" #Buca en la API la ciudad y retorna el resultado con la estructura definida
        else:
            return f"No pude obtener el clima de {ciudad}. Intenta con otro nombre de ciudad."

    except Exception as e:
        return f"Error al consultar el clima: {str(e)}" #En caso de error en la consulta, se captura la excepción y se devuelve un mensaje de error con la descripción del problema.


In [6]:
#modificamos el FinalAnswerTool para personalizar la respuesta final del agente personalizada

class MiFinalAnswerTool(FinalAnswerTool):

    #Instrcciones para el agente sobre la herramienta de respuesta final personalizada.

    """
    Versión personalizada del FinalAnswerTool.
    Agrega un prefijo, firma y conteo de caracteres a cada respuesta.
    """

    # Sobrescribimos el método forward que es el que genera la respuesta
    def forward(self, answer: str) -> str:
        #Contamos cuántos caracteres tiene la respuesta original
        num_caracteres = len(answer)

        #Construimos la respuesta con prefijo, contenido y firma
        respuesta_formateada = (
            f"🤖 Agente dice:\n\n"          # Prefijo con emoji
            f"{answer}\n\n"                  # Respuesta original del agente
            f"Esta respuesta tiene {num_caracteres} caracteres.\n"  # Conteo de caracteres
            f"--- Procesado por Angie, Cesar y Jhonatan ---"           # Firma del equipo
        )

        return respuesta_formateada

In [7]:
#Configuramos el agente con las herramientas personalizadas.

# Modelo que usará el agente (gratuito en HF Spaces)
modelo = InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct")

# Creamos el agente con las herramientas personalizadas
agente = CodeAgent(
    tools=[
        calcular_area,       # Herramienta 1: Calculador de áreas
        obtener_clima,       # Herramienta 2: Clima con API
        MiFinalAnswerTool(), # FinalAnswerTool modificado
    ],
    model=modelo,
    max_steps=5,
)

In [ ]:

#Función para responder preguntas usando el agente
def responder(pregunta):
    resultado = agente.run(pregunta) #El agente procesa la pregunta, decide qué herramientas usar y genera una respuesta.
    return resultado

interfaz = gr.Interface( #Se crea la interfaz de usuario con Gradio, donde el usuario puede ingresar una pregunta y obtener la respuesta del agente.
    fn=responder,
    inputs=gr.Textbox(
        label="¿Qué quieres preguntarle al agente?",
        placeholder="Ej: ¿Cuál es el área de un rectángulo de base 5 y altura 3?"
    ),
    outputs=gr.Textbox(label="Respuesta del Agente"),
    title="🤖 Agente Inteligente - Ejercicio 1",
    description=(
        "Agente con superpoderes:\n"
        "Puede calcular áreas de figuras geométricas\n"
        "Puede consultar el clima de cualquier ciudad\n\n"
        "Autores: Angie Giraldo | Cesar Florez | Jhonatan Rojas"
    ),
    examples=[
        ["¿Cuál es el área de un triángulo con base 6 y altura 4?"],
        ["¿Qué clima hace hoy en Medellín?"],
        ["Calcula el área de un círculo con radio 7"],
        ["¿Cómo está el tiempo en Bogotá?"],
    ]
)

if __name__ == "__main__":
    interfaz.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ¿Cuál es el área de un triángulo con base 6 y altura 4?                                                         │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  area_triangle = calcular_area(figura="triangulo", valor1=6, valor2=4)                                            
  print(area_triangle)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
El área del triángulo con base 6 y altura 4 es: 12.00

Out: None

...

[Step 2: Duration 3.48 seconds| Input tokens: 4,624 | Output tokens: 112]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  area_triangle_str = str(12.00)                                                                                   
  final_answer(area_triangle_str)                                                                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 🤖 Agente dice:

12.0

Esta respuesta tiene 4 caracteres.
--- Procesado por Angie, Cesar y Jhonatan ---

[Step 3: Duration 3.87 seconds| Input tokens: 7,192 | Output tokens: 175]